# NOVA Jarvis awaaz - 2 tap server
1. Upar menu: **Runtime > Change runtime type > T4 GPU > Save**.
2. Neeche wale **ek** cell ke left ▶ ko dabao (ya Runtime > Run all). Bas.
3. 5-15 minute baad **COPY CODE** button aayega. Use dabao.
4. NOVA kholo > Settings > JARVIS AWAAZ > **CODE PASTE (1 TAP)** > JARVIS CHECK.

Apni awaaz se bulwana ho to cell me `MY_VOICE = True` kar do (tab recording upload maangega). Warna kuch upload nahi karna.
Sirf link test (bina model): `FAKE = True`. Is page ko band mat karna.


In [ ]:
# NOVA Jarvis - sab ek cell me (GPU check, install, server, free https link, pairing code)
FAKE = False        # True = sirf beep, model nahi (link test)
MY_VOICE = False    # True = apni recording upload karke uski awaaz jaisi (sirf apni / jiska haq ho)
import os, sys, subprocess, shutil, secrets, base64, json, time, re, threading, urllib.request
if not FAKE and not shutil.which("nvidia-smi"):
    print("GPU NAHI MILA -> Runtime > Change runtime type > T4 GPU, phir dobara chalao"); raise SystemExit
TOKEN = secrets.token_urlsafe(24)
REF = ""
if not FAKE:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "huggingface_hub", "voxcpm"], capture_output=True, text=True)
    print("pip install:", "OK" if r.returncode == 0 else "FAIL\n" + r.stderr[-1500:])
    if MY_VOICE:
        from google.colab import files
        name = next(iter(files.upload()))
        REF = "/content/reference.wav"
        subprocess.run(["ffmpeg", "-y", "-i", name, "-ac", "1", "-c:a", "pcm_s16le", REF], capture_output=True)
open("/content/server.py", "wb").write(base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJOT1ZBIEphcnZpcyB2b2ljZSBzZXJ2ZXIgKHY3LjEzNikuICBXcmFwcyBGdXR1cmVQcmVzZW50TGFicy90dHMtamFydmlzIChWb3hDUE0yICsgTG9SQSBhZGFwdGVyKSBiZWhpbmQgdGhlIHRpbnkgQVBJIHRoZSBOT1ZBIGFwcCBjYWxsczoKCiAgICBQT1NUIC90dHMgICAgIGhlYWRlciAgQXV0aG9yaXphdGlvbjogQmVhcmVyIDxKQVJWSVNfVE9LRU4+ICAgYm9keSB7InRleHQiOiAiLi4uIn0gICAtPiAgMjAwIGF1ZGlvL3dhdiAoMTYgYml0IG1vbm8pCiAgICBHRVQgIC9oZWFsdGggIC0+ICB7Im9rIjogdHJ1ZSwgImVuZ2luZSI6ICJ2b3hjcG0yIiB8ICJmYWtlIn0KClB1cmUgUHl0aG9uIHN0YW5kYXJkIGxpYnJhcnkgZm9yIHRoZSB3ZWIgcGFydCAobm8gRmFzdEFQSSBuZWVkZWQpLiBUaGUgcmVhbCBlbmdpbmUgbmVlZHMgdGhlIG1vZGVsJ3Mgb3duIHBhY2thZ2VzIChzZWUgUkVBRE1FX0hJTkdMSVNILm1kKS4KCkVudmlyb25tZW50OgogICAgSkFSVklTX1RPS0VOICAgICAgIHNlY3JldCB0b2tlbi4gU3Ryb25nbHkgcmVjb21tZW5kZWQuIElmIGVtcHR5IHRoZSBzZXJ2ZXIgaXMgT1BFTiB0byBhbnlvbmUgd2hvIGZpbmRzIHRoZSBhZGRyZXNzLgogICAgSkFSVklTX1JFRkVSRU5DRSAgIHBhdGggb2YgWU9VUiByZWZlcmVuY2Uud2F2IChhIGNsZWFuIHZvaWNlIHJlY29yZGluZyB5b3UgYXJlIGF1dGhvcmlzZWQgdG8gdXNlKS4gT3B0aW9uYWwgKHY3LjEzNyk6IHdpdGhvdXQgaXQgdGhlIHJlbGVhc2UncyBvd24gcmVmZXJlbmNlIGlzIHVzZWQgaWYgaXQgaGFzIG9uZS4KICAgIEpBUlZJU19GQUtFPTEgICAgICBubyBtb2RlbDogcmV0dXJucyBhIHNob3J0IHRlc3QgdG9uZSBhcyBhIHdhdi4gT25seSBmb3IgdGVzdGluZyB0aGUgcGhvbmUgPC0+IHNlcnZlciBsaW5rLgogICAgSkFSVklTX1BPUlQgICAgICAgIGRlZmF1bHQgODAwMC4gICBKQVJWSVNfSE9TVCBkZWZhdWx0IDAuMC4wLjAuICAgSkFSVklTX01BWF9DSEFSUyBkZWZhdWx0IDgwMC4KICAgIEpBUlZJU19DRkcgLyBKQVJWSVNfU1RFUFMgLyBKQVJWSVNfU0VFRCAgIG1vZGVsIHNldHRpbmdzIChkZWZhdWx0cyAyLjQgLyA2IC8gNDIgPSB0aGUgbW9kZWwgY2FyZCdzIHNlbGVjdGVkIHJlY2lwZSkuCgpSdW46ICAgcHl0aG9uMyBzZXJ2ZXIucHkKVGhlIHJlYWwgZW5naW5lIGZvbGxvd3MgdGhlIG1vZGVsIGNhcmQncyAiVXNhZ2UiIGV4YW1wbGUuIE5PVCBURVNURUQgd2l0aCB0aGUgcmVhbCBtb2RlbCBpbiB0aGUgc2FuZGJveCB0aGF0IHdyb3RlIHRoaXMgKG5vIEdQVSwgbm8gd2VpZ2h0cykuCiIiIgppbXBvcnQgaG1hYywgaW8sIGpzb24sIG1hdGgsIG9zLCBzdHJ1Y3QsIHN5cywgdGhyZWFkaW5nLCB3YXZlCmZyb20gaHR0cC5zZXJ2ZXIgaW1wb3J0IEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIsIFRocmVhZGluZ0hUVFBTZXJ2ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVZJU0lPTiA9ICJ2NC1pbnRlcmZhY2UtMjAyNi0wOS0wNiIgICAgICAgICAgIyByZWxlYXNlIHRhZyBuYW1lZCBpbiB0aGUgbW9kZWwgY2FyZCdzIHVzYWdlIGV4YW1wbGUKUkVQTyA9ICJGdXR1cmVQcmVzZW50TGFicy90dHMtamFydmlzIgpNQVhfQ0hBUlMgPSBpbnQob3MuZW52aXJvbi5nZXQoIkpBUlZJU19NQVhfQ0hBUlMiLCAiODAwIikpCk1BWF9CT0RZID0gMjBfMDAwCkZBS0UgPSBvcy5lbnZpcm9uLmdldCgiSkFSVklTX0ZBS0UiLCAiIikgPT0gIjEiClRPS0VOID0gb3MuZW52aXJvbi5nZXQoIkpBUlZJU19UT0tFTiIsICIiKQoKX2xvY2sgPSB0aHJlYWRpbmcuTG9jaygpICAgICAgICAgICAgIyBvbmUgc3ludGhlc2lzIGF0IGEgdGltZSAob25lIEdQVSkKX2VuZ2luZSA9IHsibW9kZWwiOiBOb25lLCAic3IiOiAyNDAwMCwgInJlZiI6IE5vbmUsICJuYW1lIjogImZha2UiIGlmIEZBS0UgZWxzZSAidm94Y3BtMiJ9CgoKZGVmIGZha2Vfd2F2KHRleHQ6IHN0cikgLT4gYnl0ZXM6CiAgICAiIiJBIHNob3J0IHRlc3QgdG9uZSAobGVuZ3RoIGZvbGxvd3MgdGhlIHRleHQpLiBPbmx5IGZvciBjaGVja2luZyB0aGUgcGhvbmUgPC0+IHNlcnZlciBsaW5rLiIiIgogICAgc3IgPSAyNDAwMAogICAgbiA9IGludChzciAqIG1pbig2LjAsIDAuMjUgKyAwLjA2ICogbGVuKHRleHQpKSkKICAgIHBjbSA9IGIiIi5qb2luKHN0cnVjdC5wYWNrKCI8aCIsIGludCg4MDAwICogbWF0aC5zaW4oMiAqIG1hdGgucGkgKiAyMjAgKiBpIC8gc3IpICogbWluKDEuMCwgaSAvIDgwMCwgKG4gLSBpKSAvIDgwMCkpKSBmb3IgaSBpbiByYW5nZShuKSkKICAgIGJ1ZiA9IGlvLkJ5dGVzSU8oKQogICAgd2l0aCB3YXZlLm9wZW4oYnVmLCAid2IiKSBhcyB3OgogICAgICAgIHcuc2V0bmNoYW5uZWxzKDEpOyB3LnNldHNhbXB3aWR0aCgyKTsgdy5zZXRmcmFtZXJhdGUoc3IpOyB3LndyaXRlZnJhbWVzKHBjbSkKICAgIHJldHVybiBidWYuZ2V0dmFsdWUoKQoKCmRlZiBsb2FkX3JlYWwoKSAtPiBOb25lOgogICAgIiIiRm9sbG93cyB0aGUgbW9kZWwgY2FyZCdzIFVzYWdlIGV4YW1wbGUuIEhlYXZ5OiBydW5zIG9uY2UgYXQgc3RhcnQtdXAuIiIiCiAgICBpbXBvcnQgc291bmRmaWxlICAjIG5vcWE6IEY0MDEgIChmYWlsIGVhcmx5IGlmIG1pc3NpbmcpCiAgICBmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgc25hcHNob3RfZG93bmxvYWQKICAgIGZyb20gdm94Y3BtLmNvcmUgaW1wb3J0IFZveENQTQogICAgZnJvbSB2b3hjcG0ubW9kZWwudm94Y3BtMiBpbXBvcnQgTG9SQUNvbmZpZwoKICAgIHJlbGVhc2UgPSBQYXRoKHNuYXBzaG90X2Rvd25sb2FkKFJFUE8sIHJldmlzaW9uPVJFVklTSU9OKSkKICAgIHJlZiA9IG9zLmVudmlyb24uZ2V0KCJKQVJWSVNfUkVGRVJFTkNFIiwgIiIpCiAgICBpZiByZWYgYW5kIG5vdCBQYXRoKHJlZikuaXNfZmlsZSgpOgogICAgICAgIHN5cy5leGl0KCJKQVJWSVNfUkVGRVJFTkNFIGZpbGUgbmFoaSBtaWxpOiAlcyIgJSByZWYpCiAgICBpZiBub3QgcmVmOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgdjcuMTM3OiBubyB1cGxvYWQgbmVlZGVkLiBVc2UgYSByZWZlcmVuY2Ugc2hpcHBlZCBpbnNpZGUgdGhlIHJlbGVhc2UgaWYgdGhlcmUgaXMgb25lLCBlbHNlIG5vbmUuCiAgICAgICAgZm91bmQgPSBzb3J0ZWQocCBmb3IgcCBpbiByZWxlYXNlLnJnbG9iKCIqLndhdiIpIGlmICJyZWYiIGluIHAubmFtZS5sb3dlcigpIG9yICJyZWYiIGluIHAucGFyZW50Lm5hbWUubG93ZXIoKSkKICAgICAgICByZWYgPSBzdHIoZm91bmRbMF0pIGlmIGZvdW5kIGVsc2UgIiIKICAgIHNldHRpbmdzID0ganNvbi5sb2FkcygocmVsZWFzZSAvICJpbmZlcmVuY2VfY29uZmlnLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGJhc2UgPSBzbmFwc2hvdF9kb3dubG9hZChzZXR0aW5nc1siYmFzZV9tb2RlbCJdLCByZXZpc2lvbj1zZXR0aW5ncy5nZXQoImJhc2VfcmV2aXNpb24iKSkKICAgIGFkYXB0ZXIgPSByZWxlYXNlIC8gImFkYXB0ZXIiCiAgICByYXcgPSBqc29uLmxvYWRzKChhZGFwdGVyIC8gImxvcmFfY29uZmlnLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGxvcmFfY2ZnID0gcmF3LmdldCgibG9yYV9jb25maWciLCByYXcpIGlmIGlzaW5zdGFuY2UocmF3LCBkaWN0KSBlbHNlIHJhdwogICAgY2twdCA9IGFkYXB0ZXIgLyAic3RlcF8wMDAwMDMxIiAgICAgICAgICAgICAgICAgICAgICMgdGhlIGNhcmQgbmFtZXMgdGhpcyBjaGVja3BvaW50OyBmYWxsIGJhY2sgdG8gdGhlIGFkYXB0ZXIgZm9sZGVyCiAgICB3ZWlnaHRzID0gY2twdCBpZiBja3B0LmV4aXN0cygpIGVsc2UgYWRhcHRlcgogICAgbW9kZWwgPSBWb3hDUE0uZnJvbV9wcmV0cmFpbmVkKGhmX21vZGVsX2lkPWJhc2UsIGxvYWRfZGVub2lzZXI9RmFsc2UsIG9wdGltaXplPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvcmFfY29uZmlnPUxvUkFDb25maWcoKipsb3JhX2NmZyksIGxvcmFfd2VpZ2h0c19wYXRoPXN0cih3ZWlnaHRzKSkKICAgIF9lbmdpbmUudXBkYXRlKG1vZGVsPW1vZGVsLCBzcj1pbnQobW9kZWwudHRzX21vZGVsLnNhbXBsZV9yYXRlKSwgcmVmPXJlZikKCgpkZWYgcmVhbF93YXYodGV4dDogc3RyKSAtPiBieXRlczoKICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YKICAgIGF1ZGlvID0gX2VuZ2luZVsibW9kZWwiXS5nZW5lcmF0ZSgKICAgICAgICB0ZXh0PXRleHQsIHJlZmVyZW5jZV93YXZfcGF0aD0oX2VuZ2luZVsicmVmIl0gb3IgTm9uZSksCiAgICAgICAgY2ZnX3ZhbHVlPWZsb2F0KG9zLmVudmlyb24uZ2V0KCJKQVJWSVNfQ0ZHIiwgIjIuNCIpKSwgaW5mZXJlbmNlX3RpbWVzdGVwcz1pbnQob3MuZW52aXJvbi5nZXQoIkpBUlZJU19TVEVQUyIsICI2IikpLAogICAgICAgIHNlZWQ9aW50KG9zLmVudmlyb24uZ2V0KCJKQVJWSVNfU0VFRCIsICI0MiIpKSwgbm9ybWFsaXplPVRydWUsIGRlbm9pc2U9RmFsc2UsIG1heF9sZW49NjAwKQogICAgYnVmID0gaW8uQnl0ZXNJTygpCiAgICBzZi53cml0ZShidWYsIGF1ZGlvLCBfZW5naW5lWyJzciJdLCBmb3JtYXQ9IldBViIsIHN1YnR5cGU9IlBDTV8xNiIpCiAgICByZXR1cm4gYnVmLmdldHZhbHVlKCkKCgpkZWYgdG9rZW5fb2soaGVhZGVyOiBzdHIpIC0+IGJvb2w6CiAgICBpZiBub3QgVE9LRU46CiAgICAgICAgcmV0dXJuIFRydWUKICAgIHJldHVybiBobWFjLmNvbXBhcmVfZGlnZXN0KChoZWFkZXIgb3IgIiIpLmVuY29kZSgpLCAoIkJlYXJlciAiICsgVE9LRU4pLmVuY29kZSgpKQoKCmNsYXNzIEhhbmRsZXIoQmFzZUhUVFBSZXF1ZXN0SGFuZGxlcik6CiAgICBzZXJ2ZXJfdmVyc2lvbiA9ICJOb3ZhSmFydmlzLzEiCgogICAgZGVmIF9zZW5kKHNlbGYsIGNvZGU6IGludCwgYm9keTogYnl0ZXMsIGN0eXBlOiBzdHIpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKGNvZGUpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1UeXBlIiwgY3R5cGUpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGJvZHkpKSkKICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICBzZWxmLndmaWxlLndyaXRlKGJvZHkpCgogICAgZGVmIF9qc29uKHNlbGYsIGNvZGU6IGludCwgb2JqKSAtPiBOb25lOgogICAgICAgIHNlbGYuX3NlbmQoY29kZSwganNvbi5kdW1wcyhvYmopLmVuY29kZSgpLCAiYXBwbGljYXRpb24vanNvbiIpCgogICAgZGVmIGRvX0dFVChzZWxmKToKICAgICAgICBpZiBzZWxmLnBhdGguc3BsaXQoIj8iKVswXSA9PSAiL2hlYWx0aCI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9qc29uKDIwMCwgeyJvayI6IFRydWUsICJlbmdpbmUiOiBfZW5naW5lWyJuYW1lIl19KQogICAgICAgIHNlbGYuX2pzb24oNDA0LCB7ImRldGFpbCI6ICJub3QgZm91bmQifSkKCiAgICBkZWYgZG9fUE9TVChzZWxmKToKICAgICAgICBpZiBzZWxmLnBhdGguc3BsaXQoIj8iKVswXSAhPSAiL3R0cyI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9qc29uKDQwNCwgeyJkZXRhaWwiOiAibm90IGZvdW5kIn0pCiAgICAgICAgaWYgbm90IHRva2VuX29rKHNlbGYuaGVhZGVycy5nZXQoIkF1dGhvcml6YXRpb24iLCAiIikpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig0MDEsIHsiZGV0YWlsIjogImJhZCB0b2tlbiJ9KQogICAgICAgIHRyeToKICAgICAgICAgICAgbiA9IGludChzZWxmLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsICIwIikpCiAgICAgICAgICAgIGlmIG4gPD0gMCBvciBuID4gTUFYX0JPRFk6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig0MTMsIHsiZGV0YWlsIjogImJvZHkgc2l6ZSJ9KQogICAgICAgICAgICB0ZXh0ID0gc3RyKGpzb24ubG9hZHMoc2VsZi5yZmlsZS5yZWFkKG4pLmRlY29kZSgidXRmLTgiKSkuZ2V0KCJ0ZXh0IiwgIiIpKS5zdHJpcCgpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2pzb24oNDIyLCB7ImRldGFpbCI6ICJiYWQganNvbiJ9KQogICAgICAgIGlmIG5vdCB0ZXh0OgogICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig0MjIsIHsiZGV0YWlsIjogImVtcHR5IHRleHQifSkKICAgICAgICBpZiBsZW4odGV4dCkgPiBNQVhfQ0hBUlM6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9qc29uKDQxMywgeyJkZXRhaWwiOiAidGV4dCB0b28gbG9uZyJ9KQogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCBfbG9jazoKICAgICAgICAgICAgICAgIGRhdGEgPSBmYWtlX3dhdih0ZXh0KSBpZiBGQUtFIGVsc2UgcmVhbF93YXYodGV4dCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgIyBuZXZlciBsZWFrIGludGVybmFscyB0byB0aGUgcGhvbmUKICAgICAgICAgICAgcHJpbnQoInN5bnRoZXNpcyBlcnJvcjoiLCByZXByKGUpLCBmaWxlPXN5cy5zdGRlcnIpCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9qc29uKDUwMCwgeyJkZXRhaWwiOiAic3ludGhlc2lzIGZhaWxlZCJ9KQogICAgICAgIHNlbGYuX3NlbmQoMjAwLCBkYXRhLCAiYXVkaW8vd2F2IikKCiAgICBkZWYgbG9nX21lc3NhZ2Uoc2VsZiwgZm10LCAqYXJncyk6ICAgIyBubyByZXF1ZXN0IHRleHQgaW4gdGhlIGxvZ3MKICAgICAgICBzeXMuc3RkZXJyLndyaXRlKCIlcyAlc1xuIiAlIChzZWxmLmFkZHJlc3Nfc3RyaW5nKCksIGZtdCAlIGFyZ3MgaWYgYXJncyBlbHNlIGZtdCkpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgaWYgbm90IFRPS0VOOgogICAgICAgIHByaW50KCJXQVJOSU5HOiBKQVJWSVNfVE9LRU4gaXMgZW1wdHkgLSBhbnlvbmUgd2hvIGtub3dzIHRoaXMgYWRkcmVzcyBjYW4gdXNlIHRoZSBzZXJ2ZXIuIiwgZmlsZT1zeXMuc3RkZXJyKQogICAgaWYgbm90IEZBS0U6CiAgICAgICAgbG9hZF9yZWFsKCkKICAgIHNydiA9IFRocmVhZGluZ0hUVFBTZXJ2ZXIoKG9zLmVudmlyb24uZ2V0KCJKQVJWSVNfSE9TVCIsICIwLjAuMC4wIiksIGludChvcy5lbnZpcm9uLmdldCgiSkFSVklTX1BPUlQiLCAiODAwMCIpKSksIEhhbmRsZXIpCiAgICBwcmludCgiSmFydmlzIHNlcnZlciByZWFkeSAoZW5naW5lPSVzKSBvbiBwb3J0ICVkIiAlIChfZW5naW5lWyJuYW1lIl0sIHNydi5zZXJ2ZXJfYWRkcmVzc1sxXSkpCiAgICBzcnYuc2VydmVfZm9yZXZlcigpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="))
env = dict(os.environ, JARVIS_TOKEN=TOKEN, JARVIS_PORT="8000", JARVIS_HOST="127.0.0.1")
if FAKE: env["JARVIS_FAKE"] = "1"
if REF: env["JARVIS_REFERENCE"] = REF
log = open("/content/server.log", "w")
srv = subprocess.Popen([sys.executable, "/content/server.py"], env=env, stdout=log, stderr=log)
ok = False
for i in range(180):                    # 15 minute tak (pehli baar model download)
    if srv.poll() is not None: break
    try:
        if json.loads(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3).read())["ok"]: ok = True; break
    except Exception: pass
    time.sleep(5)
if not ok:
    print("SERVER FAIL - last log (ye mujhe bhej dena):\n" + open("/content/server.log").read()[-2500:]); raise SystemExit
subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
ADDRESS = None
t0 = time.time()
while time.time() - t0 < 60 and ADDRESS is None:
    m = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", tun.stdout.readline() or "")
    if m: ADDRESS = m.group(0)
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
if not ADDRESS:
    print("ADDRESS nahi mila (tunnel fail). Dobara chalao."); raise SystemExit
time.sleep(4)
req = urllib.request.Request(ADDRESS + "/tts", data=json.dumps({"text": "The workshop is ready."}).encode(), headers={"Authorization": "Bearer " + TOKEN, "Content-Type": "application/json"})
try: good = urllib.request.urlopen(req, timeout=120).read()[:4] == b"RIFF"
except Exception as e: good = False; print("self test error:", e)
CODE = "NOVA-JARVIS:" + base64.urlsafe_b64encode(json.dumps({"a": ADDRESS, "t": TOKEN}).encode()).decode().rstrip("=")
from IPython.display import HTML, display
print("READY" if good else "SERVER chal raha hai par awaaz test fail (upar error dekho)")
display(HTML('<button style="font-size:22px;padding:16px 24px" onclick="navigator.clipboard.writeText(\'' + CODE + '\').then(()=>{this.innerText=\'COPY HO GAYA\'}).catch(()=>{this.innerText=\'Copy nahi hua: neeche wala code long-press karke copy karo\'})">COPY CODE</button>'))
print("Agar button se copy na ho to ye poori line long-press karke copy karo:")
print(CODE)
print("Ab NOVA > Settings > JARVIS AWAAZ > CODE PASTE (1 TAP). Is page ko band mat karna.")
while True: time.sleep(60)
